# 第二版敏感性分析：p 值阈值和 exit z（只用验证集 2021–23）

其他参数固定为第二版冻结的值（10 年选股、3 年估计、entry 2.0、止损 3、最长持仓 60 天），只改变：
- **选股规则 / p 值阈值**：现在的规则（BH q=0.05，不足补到 10 对），以及"所有原始 p < 0.01 / 0.05 / 0.10 的配对都交易"（每只股票最多一对）
- **exit z**：0 / 0.25 / 0.5

只在验证集上运行，不碰测试集，冻结的参数也不改。需要先跑完 `selection_window_study_v2.ipynb`。

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
import selection_window_study as sws

sws.PAIR_UNIVERSE = "sub_industry"
sws.MIN_DOLLAR_HEDGE = 0.25
OUT = Path("selection_window_study_outputs_v2")
prices = sws.load_prices()
p = json.loads((OUT / "chosen_parameters.json").read_text())
print("冻结的参数:", p)

In [ ]:
screen = pd.read_csv(OUT / f"screen_cut2020_{p['s']}Y.csv").sort_values("P Value")

def unique_pairs(df):
    used, out = set(), []
    for a, b in zip(df["Ticker A"], df["Ticker B"]):
        if a in used or b in used:
            continue
        used |= {a, b}
        out.append((a, b))
    return out

current = pd.read_csv(OUT / f"validation_pairs_{p['s']}Y.csv")
rules = {"Current: BH q=0.05, top-up to 10": list(zip(current["Ticker A"], current["Ticker B"]))}
for t in (0.01, 0.05, 0.10):
    rules[f"All pairs with raw p < {t}"] = unique_pairs(screen[screen["P Value"] < t])
for k, v in rules.items():
    print(k, "->", len(v), "pairs")

In [ ]:
rows = []
for name, pairs in rules.items():
    for exit_z in (0.0, 0.25, 0.5):
        port, log, pr = sws.run_strategy(prices, pairs, [2021, 2022, 2023],
                                         p["n"], p["entry"], exit_z, p["stop"], p["hold"], cost=0.001)
        rows.append({"Rule": name, "Pairs": len(pairs), "Exit z": exit_z, **sws.summarize(port, log, pr)})
sens = pd.DataFrame(rows)
sens.to_csv(OUT / "sensitivity_pthreshold_exit.csv", index=False)
display(sens[["Rule", "Pairs", "Exit z", "Sharpe", "Annualized Return (%)", "Max Drawdown (%)", "Trades", "Win Rate (%)"]].round(2))
display(sens.pivot(index="Rule", columns="Exit z", values="Sharpe").round(2))